[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oceanmodeling/xarray-selafin/blob/main/workshop/01_getting_started_colab.ipynb)

*Run this notebook interactively in your browser via the Colab link above (no installation required).*

<img src="https://opentelemac.org/api/assets/b83a3859-88f7-4e40-8e7f-a9eff92853aa" alt="opentelemac" width="400"/>
<img src="https://docs.xarray.dev/en/v2023.10.1/_static/dataset-diagram-logo.png" alt="xarray" width="400"/>

# xarray-selafin 101

[TELEMAC](https://www.opentelemac.org) Selafin (`.slf`) files read as `xarray.Dataset`

What you'll learn:
1. open a Selafin file (1D, 2D, 3D)
2. understand the dataset layout: `time` / `plan` / `node`, `x`/`y`, `ikle2`, `ipobo`
3. slice on time and nodes (index, date, location, bounding box)
4. compute a new variable and write it back to Selafin

## 0. Setup

- **Data:** read-only. Defaults to the repo's `tests/data`; override it with `XSLF_DATA`.
- **Scratch:** your own writable folder. Defaults to `$TMPDIR/xarray-selafin-workshop/$USER`; override it with `XSLF_SCRATCH`.

In [ ]:
import getpass
import os
import tempfile
from pathlib import Path

import holoviews as hv
import hvplot.xarray
import numpy as np
import xarray as xr

from xarray_selafin.xarray_backend import SelafinAccessor  # needed to write Selafin

hv.extension("bokeh")

DATA = Path(os.environ.get("XSLF_DATA", "../tests/data")).resolve()
SCRATCH = Path(os.environ.get("XSLF_SCRATCH", Path(tempfile.gettempdir()) / "xarray-selafin-workshop" / getpass.getuser()))
SCRATCH.mkdir(parents=True, exist_ok=True)

print("data   :", DATA)
print("scratch:", SCRATCH)
print("xarray :", xr.__version__)
print("engines:", [e for e in xr.backends.list_engines() if e == "selafin"])
sorted(p.name for p in DATA.glob("*.slf"))


## 1. Open files

Normally the `engine="selafin"` is automatically inferred; the backend registers itself through an xarray entry point. Data is **lazy**: nothing is read from disk until you ask for values.

### 1.1 Open a 1D file

In [ ]:
ds = xr.open_dataset(DATA / "r1d_tomsail.slf")#, engine="selafin")
ds

### 1.2 Open a 2D file

In [ ]:
ds = xr.open_dataset(DATA / "r2d_tidal_flats.slf")#, engine="selafin")
ds

### 1.3 Open a 3D file

In [ ]:
ds = xr.open_dataset(DATA / "r3d_bump.slf")#, engine="selafin")
ds

### Things to notice

| | 1D (`r1d_tomsail`) | 2D (`r2d_tidal_flats`) | 3D (`r3d_bump`) |
|---|---|---|---|
| dims | `(time, node)` | `(time, node)` | `(time, plan, node)` |
| coords | `x`, `y` on `node` | `x`, `y` on `node` | `x`, `y` on `node` (same for every `plan`) |
| variables | all `(time, node)` | all `(time, node)` | all `(time, plan, node)`, incl. `Z` (elevation of each level) |
| `time` | starts at the file's `date_start`: 2023-09-01 | no `date_start` in the file: defaults to 1900-01-01 | no `date_start` in the file: defaults to 1900-01-01 |
| `attrs["variables"]` | maps each short name to its Selafin `(long name, unit)` | same | same |
| `attrs["ikle2"]` | `(nnode, 1)`: every point is its own element | `(nelem, 3)`: triangle connectivity | `(nelem, 3)`: connectivity of the 2D base mesh |
| `attrs["ikle3"]` | absent | absent | `(nelem * (nplan - 1), 6)`: prisms, indexing the 3D node numbering (`nnode * nplan`) |
| `attrs["ipobo"]` | `1..nnode`: **all** points are boundary points | length `nnode`: `0` = interior node, `n > 0` = boundary node, `n` being its rank along the boundary | length `nnode * nplan`: same convention, repeated for each `plan` |

Connectivity tables (`ikle2`, `ikle3`) are **1-based** (TELEMAC/Fortran convention): subtract 1 before using them as numpy indices.


In [ ]:
ikle2 = ds.attrs["ikle2"]
ikle3 = ds.attrs["ikle3"]
print("triangles:", ikle2.shape, "| min index:", ikle2.min(), "| max index:", ikle2.max(), "| nodes:", ds.sizes["node"])
print("prisms:   ", ikle3.shape, "| min index:", ikle3.min(), "| max index:", ikle3.max(), "| 3D nodes:", ds.sizes["node"] * ds.sizes["plan"])

## 2. Quick look at the mesh

The mesh is unstructured: no grid, just a cloud of nodes. The quickest view is a scatter of the nodes colored by a variable — here the bed elevation `B` — using `hvplot` (interactive, bokeh backend). Heavier rendering with datashader is covered in the advanced notebook.

In [ ]:
ds = xr.open_dataset(DATA / "r2d_tidal_flats.slf")#, engine="selafin")

ds.hvplot.scatter(x="x", y="y", c="B", cmap="fire").opts(width=1200, height=120)

Adding `hover_cols="node"` allows you to identify the node number:

In [ ]:
ds.hvplot.scatter(x="x", y="y", c="B", cmap="fire", hover_cols="node").opts(width=1200, height=120)

Check one node:

In [ ]:
ds.isel(node=352).hvplot(grid=True).opts(width=1000, height=300, title="all variables")

In [ ]:
ds.S.isel(node=352).hvplot(grid=True).opts(width=1000, height=300, title="S only")

## 3. Slicing on time

Selafin data is just xarray data, so use `isel` (by position) and `sel` (by label).

In [ ]:
ds = xr.open_dataset(DATA / "r2d_tidal_flats.slf")
ds.isel(time=-1).time                                   # last frame
ds.isel(time=slice(0, 5)).time                          # first 5 frames
ds.sel(time="1900-01-01T12:00").time                    # returns empty
ds.sel(time="1900-01-01T12:00", method="nearest").time  # returns nearest frame

## 4. Slicing on nodes

The mesh is unstructured: there is no `x`/`y` grid, only a list of nodes. To select by space, compute an index first.

In [ ]:
# by index
ds.isel(node=[0, 10, 100])

# nearest node to a point
x0, y0 = 0.0, 0.0
inode = int(((ds.x - x0) ** 2 + (ds.y - y0) ** 2).argmin(dim="node"))
print("nearest node:", inode, float(ds.x[inode]), float(ds.y[inode]))

ds.S.isel(node=inode).hvplot()

In [ ]:
# bounding box -> node indices
in_box = ((ds.x > 0) & (ds.x < 10_000) & (ds.y > 0)).values
ds_box = ds.isel(node=np.flatnonzero(in_box))
ds_box.hvplot.scatter(x="x", y="y", c="B", cmap="fire", hover_cols="node").opts(width=1200, height=120)

⚠️ **Gotcha:** after a node subset, `attrs["ikle2"]` still refers to the *original* node numbering. The subset is fine for analysis (time series, stats), but don't plot it as a mesh or write it to Selafin without renumbering the connectivity first.

## 5. Reduce in time

Standard xarray reductions work on lazy data.

In [ ]:
ds = xr.open_dataset(DATA / "r2d_tidal_flats.slf")
s_max = ds.S.max("time")
s_range = ds.S.max("time") - ds.S.min("time")

s_range.hvplot.scatter(x="x", y="y", c="S", cmap="fire", hover_cols="node", title="max tidal range").opts(width=1200, height=120)

## 6. 3D files: the `plan` dimension

In 3D the dims are `(time, plan, node)`: `plan` holds the vertical levels (0 = bottom), and `Z` is the elevation of each level.

In [ ]:
ds = xr.open_dataset(DATA / "r3d_bump.slf", engine="selafin")
ds.hvplot.scatter(x="x", y="Z", c="U", hover_cols="plan", cmap="fire", title="U in the (x, Z) section").opts(width=1200, height=300, yformatter="%.1f")

### 6.1 Convert 3D to 2D files

In [ ]:
ds_bottom = ds.selafin.get_dataset_as_2d(plan=0)
ds_top = ds.selafin.get_dataset_as_2d(plan=ds.sizes["plan"] - 1)  # can't use -1: no negative indices here (to be added in the future)
ds_top

## 7. Compute and write back to Selafin

Add a variable, register its Selafin `(long name, unit)` in `attrs["variables"]`, and write to **your scratch folder**. Without that entry, the writer falls back to TELEMAC's known IDs or to the bare name with unit `?`.

In [ ]:
ds["SPEED"] = (("time", "plan", "node"), np.sqrt(ds.U**2 + ds.V**2 + ds.W**2).data)
ds.hvplot.scatter(x="x", y="Z", c="SPEED", hover_cols="plan", cmap="fire").opts(width=1200, height=300)

ds.attrs["variables"]["SPEED"] = ("SCALAR VELOCITY", "M/S")

fout = SCRATCH / "bump_speed.slf"
ds.selafin.write(fout)
f"saved in {fout}"

## 8. Exercise

1. load the `tom_manche.slf` Selafin (located in `DATA / "tom_manche.slf"`)
2. visualize the significant wave height (`HAUTEUR_HM0`)
3. add the OSM basemap: convert the dataset to a dataframe (`df = ds.to_dataframe().reset_index()`, necessary to plot on a map with geoviews) and use `geo=True, tiles=True`
4. plot the vector field instead of the points, using `hvplot.vectorfield` (magnitude `HAUTEUR_HM0`, angle `TETA_MOYEN`)

Solutions: [01_getting_started_solutions.ipynb](01_getting_started_solutions.ipynb)

In [ ]:
# your code here
# 1.
# ...

# 2.
# ds.hvplot.scatter(...)

# 3.
# import hvplot.pandas
# df = ds.to_dataframe().reset_index()  # necessary to visualize with geoviews
# df.hvplot.points(..., geo=True, tiles=True, ...)

# 4.
# df.hvplot.vectorfield(..., mag=..., angle=..., geo=True, tiles=True, ...)